[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C29_Frontier_Interp_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用**合成叠加数据**与 **numpy toy transformer** 模拟前沿 interp 机制，再对拍 **ground truth**。

这个 notebook 做三件事：① 确认环境；② 用一个最小例子体会**叠加（superposition）**为什么让「读单个神经元」失效（本课全部动机）；③ 立下全课纪律——**在已知真相的合成数据上对拍验证**。

> 本课是 **C06 基础 interp 的前沿深化**。probe / activation patching / logit lens 在 C06；这里从叠加直奔 SAE / 特征 / 电路前沿 / steering / model diffing。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画帕累托前沿 / 激活直方图）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 叠加：为什么不能读单个神经元

构造一个最小的叠加场景：在 `d` 维空间里塞 `m > d` 个**真特征方向**，每个输入只**稀疏地**激活其中少数几个特征，把它们线性叠加成激活向量 `x`。这模拟了真实 residual stream：特征数远多于维度。

然后我们问：**单个神经元（坐标轴）能区分一个特征吗？** 答案是「不能干净地区分」——这就是**多义性（polysemanticity）**，也是本课要用 SAE 解决的根本问题。

In [ ]:
rng = np.random.default_rng(0)
d, m = 8, 20                 # 8 维空间，硬塞 20 个特征 -> 必然叠加
# 真特征方向：随机单位向量（非正交，因为 m>d 根本无法两两正交）
F = rng.standard_normal((m, d))
F = F / np.linalg.norm(F, axis=1, keepdims=True)   # 每行(每个特征)单位化

# 任意两个特征方向的夹角余弦：理想正交应≈0，叠加下必然偏离0
G = F @ F.T
offdiag = G[~np.eye(m, dtype=bool)]
print(f'd={d}, m={m}  (m>d 故无法正交)')
print(f'特征两两 |cos| 均值 = {np.abs(offdiag).mean():.3f}  (>0 即有干扰/串扰)')
assert m > d
assert np.abs(offdiag).mean() > 0.05, '叠加下特征方向必然非正交 -> 有串扰'
print('✅ m>d 时特征方向无法两两正交，彼此干扰 —— 这是叠加的本质')

现在生成稀疏激活的数据，并验证「读单个神经元」会把多个特征混在一起（多义性）。

In [ ]:
def make_superposed(n, F, p_active=0.15, noise=0.02, rng=rng):
    '''每个样本以概率 p_active 独立激活每个特征，激活幅度~U(0.5,1.5)，线性叠加+噪声。
       返回 (激活 X[n,d], 真特征激活码 S[n,m])。S 是 ground truth：谁被激活了。'''
    m_, d_ = F.shape
    S = (rng.random((n, m_)) < p_active).astype(float)        # 0/1 是否激活
    S *= rng.uniform(0.5, 1.5, size=(n, m_))                  # 激活幅度
    X = S @ F + noise * rng.standard_normal((n, d_))          # 叠加 + 噪声
    return X, S

X, S = make_superposed(4000, F)
print('激活 X:', X.shape, '| 真特征码 S:', S.shape, '| 平均每样本激活特征数 L0 =', S.astype(bool).sum(1).mean().round(2))

# 看神经元 0（坐标轴 0）：它对哪些真特征响应？(neuron 值 与 各特征激活 的相关)
neuron0 = X[:, 0]
corr = np.array([np.corrcoef(neuron0, S[:, j])[0, 1] for j in range(m)])
strong_feats = np.where(np.abs(corr) > 0.15)[0]
print(f'\n神经元0 与 >1 个真特征强相关：特征 {list(strong_feats)} (|corr|>0.15)')
assert len(strong_feats) >= 2, '单个神经元应对多个特征响应 = 多义'
print('✅ 单个神经元是【多义】的：它把多个无关特征混在一起，无法解释 —— 所以需要 SAE（模块 01）')

## 3 · 立纪律：在已知真相上对拍

本课每个机制都在**我们植入了 ground truth 的合成数据**上验证。上面的 `S`（谁被激活）与 `F`（真特征方向）就是真相。

预演模块 01 的核心检验：一个**理想**的特征提取器，应该能从 `X` 把真激活 `S` 读回来。这里先用「已知 F 时的最小二乘解码」当**上界参照**（真 SAE 不知道 F，要自己学），确认数据本身是可解的。

In [ ]:
# 已知真特征 F 时，用最小二乘从 X 恢复特征激活：S_hat = X @ pinv(F)
F_pinv = np.linalg.pinv(F)          # (d, m)
S_hat = X @ F_pinv                  # (n, m) 对每个特征的恢复

# 逐特征看恢复质量（与真 S 的相关），作为「可恢复性」上界
rec_corr = np.array([np.corrcoef(S_hat[:, j], S[:, j])[0, 1] for j in range(m)])
# 随机方向基线：与真激活码应几乎不相关(~0)
rand_dir = rng.standard_normal((d, m)); rand_dir /= np.linalg.norm(rand_dir, axis=0, keepdims=True)
S_rand = X @ rand_dir
rand_corr = np.array([np.corrcoef(S_rand[:, j], S[:, j])[0, 1] for j in range(m)])
print(f'已知F时逐特征恢复相关：均值={rec_corr.mean():.3f}  最小={rec_corr.min():.3f}')
print(f'随机方向基线相关    ：均值={np.abs(rand_corr).mean():.3f} (应≈0)')
assert rec_corr.mean() > np.abs(rand_corr).mean() + 0.2, '用真特征方向应远胜随机方向'
assert rec_corr.mean() > 0.5, '已知真特征方向时应能恢复出明显信号（数据可解）'
print('✅ 数据可解：已知 F 能把真特征激活读回来（远胜随机方向）。')
print('   注意 m>d 的叠加使串扰把恢复压在 ~0.6（非 1.0）—— 这正是叠加的代价。')
print('   模块 01 的挑战是：SAE 【不知道 F】，要从 X 无监督地把这组特征学出来。')

## 4 · 一个会贯穿全课的对拍工具

把「对拍」封装成统一裁判，后面每个模块都用它判定「我的实现 == 参考/真相」。它就是全课所有 `assert` 背后的统一标准。

In [ ]:
def check_close(name, got, ref, atol=1e-8, rtol=1e-5):
    '''对拍：被测结果 vs 参考/真相。打印最大误差并 assert。'''
    got = np.asarray(got, dtype=float); ref = np.asarray(ref, dtype=float)
    ok = np.allclose(got, ref, atol=atol, rtol=rtol)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

# 演示：手写 ReLU 对拍 numpy
z = rng.standard_normal((5, 5))
check_close('relu vs np.maximum', np.where(z > 0, z, 0.0), np.maximum(z, 0.0))
print('\n这就是全课工作流：写机制 -> 对拍参考/真相 -> assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个 SAE / 归因 / steering / 检测器，都会在**植入了真相的合成数据**上对拍验证；结构正确则数值合理，逻辑可迁移到 TransformerLens / SAELens 等真工具。

**接下来五个模块**：01 SAE → 02 单义特征与叠加 → 03 电路与 Attribution → 04 特征 Steering → 05 可解释性用于安全。每一步都建立在前一步之上。

下一站：**模块 01 · 稀疏自编码器 SAE** —— 把叠加解压成单义特征。